# JS09: Lab 4

Perbandingan Annoy, FAISS, dan HNSW pada 1.000.000 data berdimensi lima. Tunggu sampai pembangunan setiap indeks selesai.

## Instalasi library

In [1]:
%pip install -q annoy==1.17.3 faiss-cpu==1.13.2 hnswlib==0.8.0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\syahr\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


## Import library

In [2]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

## Membuat dataset dan query

In [3]:
n_data = 1_000_000
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

query = np.random.random((1, dim)).astype(np.float32)
k = 10

## Annoy

In [4]:
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

=== Annoy ===


Build time: 9.53767728805542 detik
Query time: 0.00017762184143066406 detik
Neighbors: [269148, 929890, 71228, 925267, 930798] ...


## FAISS IndexFlatL2

In [5]:
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")


=== FAISS (IndexFlatL2) ===
Build time: 0.005603313446044922 detik
Query time: 0.0044329166412353516 detik
Neighbors: [269148 929890  71228 925267 930798] ...


## HNSW

In [6]:
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")


=== HNSW (hnswlib) ===


Build time: 49.39532160758972 detik
Query time: 0.00020265579223632812 detik
Neighbors: [269148 929890  71228 925267 930798] ...


Build time mengukur pembangunan indeks, sedangkan query time mengukur pencarian sepuluh tetangga. Lima indeks pertama ditampilkan seperti pada panduan. IndexFlatL2 merupakan pencarian exact. Indeks tetangga dan waktu dapat berubah karena data dibuat secara acak tanpa seed.